# 03 — Análise Estrutural Avançada (Parte III)

**Aluno:** Vitor Renato Santos dos Santos  
**Matrícula:** 222217109 — UFBA  
**Dataset:** ego-Gplus (Google+ social circles) — SNAP  
**Grafo atribuído:** G2 ($n = 107.614$, $m = 12.238.285$, componente conexa única)

---

### Objetivos deste Notebook:
1. **Análise de Mundo Pequeno (*Small-World* — Apêndice A):**
   - Avaliar as propriedades métricas de Watts–Strogatz ($L$ e $C$) do grafo real $G$.
   - Comparar com uma rede aleatória equivalente $G(n, m)$ de Erdős–Rényi de duas formas:
     - *(a)* **Teórica:** $L_{\text{rand}} \approx \ln(n)/\ln(\langle k \rangle)$, $C_{\text{rand}} \approx \langle k \rangle / n$.
     - *(b)* **Empírica:** Geração manual em Numba (`@njit`) de $G(n, m)$ com $n=107.614$ e $m=12.238.285$, mensurando APL amostrado ($200$ fontes) e clusterização média manual via dois ponteiros.
   - Calcular as razões $L/L_{\text{rand}}$, $C/C_{\text{rand}}$ e o coeficiente de mundo pequeno $\sigma = (C/C_{\text{rand}})/(L/L_{\text{rand}})$, interpretando as implicações para difusão social.
2. **Análise de Lei de Potência e Diagnóstico de Cauda (Apêndice B):**
   - Visualizar a densidade empírica $P(k)$ e a CCDF $P(K \ge k)$ em escala log-log com modelos sobrepostos.
   - Implementar a metodologia rigorosa de Clauset, Shalizi & Newman (2009): varredura sistemática de $k_{\min}$ minimizando a distância de Kolmogorov-Smirnov ($D_{\text{KS}}$).
   - Confrontar a cauda observada contra distribuições alternativas concorrentes (**Log-Normal** e **Exponencial**) via Máxima Verossimilhança (MLE), Critério de Informação de Akaike (AIC) e Razão de Log-Verossimilhança.
   - Diagnosticar com rigor estatístico a presença de curvatura acentuada e corte para graus elevados ($k > 10^3$).
3. **Análise de Robustez Topológica e Tolerância a Falhas (Apêndice C):**
   - Avaliar o impacto da perda de $5\%$ dos vértices ($r = \lceil 0,05 \cdot n \rceil = 5.381$ nós) sob três regimes experimentais:
     - *(i)* **Falha aleatória:** $T = 30$ repetições independentes com semente fixa.
     - *(ii)* **Ataque direcionado por Grau:** remoção dos $r$ nós mais conectados (*hubs*).
     - *(iii)* **Ataque direcionado por Betweenness Centrality:** remoção dos $r$ nós com maior intermediação (amostragem com $200$ pivôs via Brandes compilado em Numba).
   - Quantificar as quatro métricas obrigatórias: Fração da GCC ($S$), Total de Componentes Conexas ($B$), APL e Diâmetro na GCC ($C$) e Fração de Nós Isolados ($D$), comparando com o Grafo Base intacto.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, display
except ImportError:
    class Image:
        def __init__(self, filename):
            self.filename = filename
        def __repr__(self):
            return f"[Image: {self.filename}]"
    display = print

os.environ["MPLCONFIGDIR"] = os.path.abspath(".matplotlib_cache")
os.makedirs(os.environ["MPLCONFIGDIR"], exist_ok=True)

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

from src.graph_loader import load_csr_graph

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

RESULTS_DIR = "results"
FIGURES_DIR = "figures"

print("Ambiente configurado com sucesso para a Parte III!")

## 1. Small-World: Modelo de Watts–Strogatz (Apêndice A)

Uma rede é classificada como de **mundo pequeno** (*small-world*) segundo o critério formal de Watts e Strogatz (1998) se apresentar simultaneamente:
1. **Caminhos mínimos curtos:** $L(G) \approx L_{\text{rand}}$, crescendo logarithmicamente com o número de vértices ($\ln(n)/\ln(\langle k \rangle)$);
2. **Forte agrupamento local:** $C(G) \gg C_{\text{rand}}$, ordens de magnitude superior ao de uma rede aleatória equivalente;
3. **Índice de Mundo Pequeno elevado:** $\sigma = \frac{C / C_{\text{rand}}}{L / L_{\text{rand}}} \gg 1$.

Para testar rigorosamente a hipótese, confrontamos os valores empíricos consolidados de $G$ ($L = 2,8603$, $C = 0,490118$) contra o grafo aleatório de Erdős–Rényi $G(n, m)$ tanto por aproximação teórica quanto por **simulação empírica direta** gerada em Numba com os mesmos $n = 107.614$ e $m = 12.238.285$.

In [ ]:
SW_PATH = os.path.join(RESULTS_DIR, "parte3_small_world.json")
with open(SW_PATH, "r", encoding="utf-8") as f:
    sw_data = json.load(f)

sw_rows = [
    {
        "Métrica": "Comprimento Médio dos Caminhos (APL, L)",
        "Grafo Real G (ego-Gplus)": f"{sw_data['grafo_real']['L_G']:.4f}",
        "ER Teórico": f"{sw_data['teorico']['L_rand']:.4f}",
        "ER Empírico (G(n, m) Numba)": f"{sw_data['empirico']['L_rand']:.4f}",
        "Razão (G / ER Empírico)": f"{sw_data['empirico']['ratio_L']:.4f} (~1x)"
    },
    {
        "Métrica": "Coeficiente de Clusterização Médio (C)",
        "Grafo Real G (ego-Gplus)": f"{sw_data['grafo_real']['C_G']:.6f}",
        "ER Teórico": f"{sw_data['teorico']['C_rand']:.6f}",
        "ER Empírico (G(n, m) Numba)": f"{sw_data['empirico']['C_rand']:.6f}",
        "Razão (G / ER Empírico)": f"{sw_data['empirico']['ratio_C']:.2f}x (>> 1)"
    },
    {
        "Métrica": "Índice de Mundo Pequeno (sigma)",
        "Grafo Real G (ego-Gplus)": "—",
        "ER Teórico": f"{sw_data['teorico']['sigma']:.2f}",
        "ER Empírico (G(n, m) Numba)": f"{sw_data['empirico']['sigma']:.2f}",
        "Razão (G / ER Empírico)": f"sigma = {sw_data['empirico']['sigma']:.2f} >> 1"
    }
]
df_sw = pd.DataFrame(sw_rows)
display(df_sw)

### Visualização Comparativa da Propriedade Small-World

O gráfico abaixo ilustra o contraste categórico: enquanto as distâncias médias geodésicas se mantêm praticamente equivalentes ($L(G)/L_{\text{rand}} \approx 1,09$), a densidade de triângulos locais da rede social é mais de **232 vezes maior** que a de uma rede aleatória pura ($C(G)/C_{\text{rand}} \approx 232,08$).

In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, "parte3_small_world_comparacao.png")))

> **Conclusão e Implicação Prática (Pergunta 1):**  
> A rede ego-Gplus satisfaz cabalmente a condição de *small-world* ($\sigma_{\text{emp}} \approx 212,48 \gg 1$).  
> Na prática, isso traduz a estrutura modular das redes sociais: os relacionamentos formam **círculos sociais densamente conectados** (amigos de amigos compartilham laços mútuos com alta probabilidade, favorecendo confiança e compartilhamento redundante), ao passo que um número relativamente pequeno de conexões inter-comunitárias (*weak ties*) atua como pontes que garantem que qualquer informação, rumor ou influência alcance a totalidade dos $107.614$ indivíduos em menos de 3 passos médios.

---
## 2. Lei de Potência e Diagnóstico de Cauda (Apêndice B)

Para determinar se o grafo apresenta distribuição livre de escala (*scale-free*), isto é, $P(k) \propto k^{-\gamma}$, empregamos a metodologia formal de **Clauset, Shalizi e Newman (2009)**:
1. **Estimador de Máxima Verossimilhança (MLE)** para o expoente contínuo discretizado:
   $$\hat{\gamma} = 1 + n_{\text{tail}} \left[ \sum_{i=1}^{n_{\text{tail}}} \ln \left( \frac{k_i}{k_{\min} - 0,5} \right) \right]^{-1}$$
2. **Varredura de $k_{\min}$:** minimização da distância de Kolmogorov–Smirnov ($D_{\text{KS}} = \max |S(k) - P(k)|$) entre a CDF empírica e a teórica.
3. **Confronto de Modelos Alternativos:** comparação da cauda observada contra a distribuição **Log-Normal Truncada** e a **Exponencial Truncada**, avaliando a Razão de Log-Verossimilhança ($R$) e o Critério de Informação de Akaike ($\text{AIC} = 2p - 2\ln L$).

In [ ]:
PL_PATH = os.path.join(RESULTS_DIR, "parte3_power_law.json")
with open(PL_PATH, "r", encoding="utf-8") as f:
    pl_data = json.load(f)

opt_c = pl_data["otimo_clauset"]
ref_1500 = pl_data["referencia_1500"]

tail_rows = [
    {
        "Limiar de Cauda (k_min)": f"k_min = {opt_c['k_min']} (Mínimo KS Clauset)",
        "Nós na Cauda": f"{opt_c['n_tail']:,} ({opt_c['fraction_tail_pct']:.2f}%)",
        "Power Law (gamma, KS)": f"gamma = {opt_c['power_law']['gamma']:.4f} (KS = {opt_c['power_law']['ks']:.4f})",
        "Log-Normal (mu, sigma, KS)": f"mu = {opt_c['log_normal']['mu']:.2f}, s = {opt_c['log_normal']['sigma']:.2f} (KS = {opt_c['log_normal']['ks']:.4f})",
        "Exponencial (scale, KS)": f"scale = {opt_c['exponencial']['scale']:.2f} (KS = {opt_c['exponencial']['ks']:.4f})",
        "AIC (PL vs LN)": f"PL: {opt_c['power_law']['aic']:.0f} | LN: {opt_c['log_normal']['aic']:.0f} (Delta = {opt_c['power_law']['aic'] - opt_c['log_normal']['aic']:.1f})"
    },
    {
        "Limiar de Cauda (k_min)": f"k_min = {ref_1500['k_min']} (Regime Típico de Cauda)",
        "Nós na Cauda": f"{ref_1500['n_tail']:,} ({ref_1500['fraction_tail_pct']:.2f}%)",
        "Power Law (gamma, KS)": f"gamma = {ref_1500['power_law']['gamma']:.4f} (KS = {ref_1500['power_law']['ks']:.4f})",
        "Log-Normal (mu, sigma, KS)": f"mu = {ref_1500['log_normal']['mu']:.2f}, s = {ref_1500['log_normal']['sigma']:.2f} (KS = {ref_1500['log_normal']['ks']:.4f})",
        "Exponencial (scale, KS)": f"scale = {ref_1500['exponencial']['scale']:.2f} (KS = {ref_1500['exponencial']['ks']:.4f})",
        "AIC (PL vs LN)": f"PL: {ref_1500['power_law']['aic']:.0f} | LN: {ref_1500['log_normal']['aic']:.0f} (Delta = {ref_1500['power_law']['aic'] - ref_1500['log_normal']['aic']:.1f})"
    }
]
df_tail = pd.DataFrame(tail_rows)
display(df_tail)

### Curva de Otimização de Clauset e Ajuste Gráfico de Cauda

A figura abaixo exibe a varredura da distância $D_{\text{KS}}$ em função de $k_{\min}$ e a sobreposição das curvas teóricas sobre a PDF e a CCDF empíricas em escala logarítmica.

In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, "parte3_power_law_ks_scan.png")))
display(Image(filename=os.path.join(FIGURES_DIR, "parte3_distribuicao_graus_ajuste.png")))

> **Conclusão Honesta sobre a Lei de Potência (Pergunta 2):**  
> A rede **não segue uma lei de potência pura**. A inspeção visual da CCDF revela curvatura sistemática com declínio abrupto para graus elevados ($k > 10^3$).  
> A otimização formal de Clauset só atinge seu mínimo em $k_{\min} = 4.496$, onde o expoente estimado $\gamma = 4,56$ cai nitidamente fora do intervalo plausível de redes livres de escala ($2 < \gamma < 3$) e a cauda engloba míseros $0,30\%$ dos nós.  
> No regime em que os super-hubs da rede de fato se situam ($k \ge 1.500$), a distribuição **Log-Normal supera a Lei de Potência com folga estatística incontestável** ($\text{KS} = 0,0099$ contra $0,0652$; $\Delta\text{AIC} = 213,8$ a favor da Log-Normal). Logo, o ego-Gplus é caracterizado por uma distribuição de cauda pesada de corpo log-normal com saturação/corte exponencial, e não por anexo preferencial irrestrito.

---
## 3. Robustez Topológica e Tolerância a Ataques (Apêndice C)

Para testar a resiliência da rede, removemos uma fração controlada de $5\%$ dos nós ($r = \lceil 0,05 \cdot 107.614 \rceil = 5.381$ vértices) sob três cenários:
1. **Falha Aleatória ($T = 30$ repetições):** remoção de nós selecionados uniformemente ao acaso.
2. **Ataque Direcionado por Grau (*Degree Centrality*):** remoção dos $5\%$ vértices de maior grau ($k_u$).
3. **Ataque Direcionado por Intermediação (*Betweenness Centrality*):** remoção dos $5\%$ vértices com maior centralidade de intermediação aproximada ($200$ pivôs via Brandes).

Todas as rotinas operaram sobre matrizes CSR com máscaras booleanas compiladas em Numba (`@njit`), sem duplicar ou reconstruir a estrutura matricial.

In [ ]:
TABELA_COMP_PATH = os.path.join(RESULTS_DIR, "parte3_tabela_comparativa.csv")
df_comp = pd.read_csv(TABELA_COMP_PATH)
display(df_comp)

### Visualização dos 4 Painéis de Robustez (Boxplots e Marcadores de Ataque)

A figura abaixo sintetiza os 4 indicadores de robustez topológica solicitados pelo enunciado:
- **Painel A:** Fração do Maior Componente ($S = |\text{GCC}|/n$);
- **Painel B:** Número Total de Componentes Conexas ($B$);
- **Painel C:** Comprimento Médio dos Caminhos na GCC ($C$);
- **Painel D:** Fração de Nós Isolados ($D$).

In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, "parte3_robustez_boxplots.png")))

> **Interpretação e Conclusão de Robustez (Pergunta 3):**  
> A rede exibe extrema **robustez assimétrica**, confirmando a assinatura típica de redes com nós altamente heterogêneos.  
> - **Sob falha aleatória:** a componente gigante permanece praticamente incólume ($S_{\text{rand}} = 94,77\%$, cobrindo $99,76\%$ dos nós sobreviventes), a fragmentação é diminuta (média de $245,6$ componentes formadas, das quais $244,1$ são nós isolados) e a distância geodésica média se preserva em $2,8397$.  
> - **Sob ataque direcionado:** a remoção dos $5\%$ nós centrais causa desarticulação aguda. O ataque por grau fragmenta a rede em **$6.890$ componentes** e reduz a GCC para $88,51\%$. Mais destrutivo ainda, o ataque por intermediação (*betweenness*) pulveriza a rede em **$10.582$ componentes**, elevando o número de nós isolados para $10.438$ ($10,21\%$ dos nós ativos), dobrando o diâmetro da GCC de $6$ para $14$ e aumentando o APL para $3,7709$.  
> Essa profunda discrepância ($S_{\text{rand}} \gg S_{\text{cent}}$) decorre do fato de que a coesão macroscópica de círculos sociais distintos depende criticamente de um núcleo restrito de conexões estratégicas e indivíduos altamente populares.

---
## 4. Resumo Executivo e Tabela Geral de Conformidade da Parte III

Abaixo consolidamos as respostas finais estruturadas e a rastreabilidade metodológica de cada entrega da Parte III.

In [ ]:
RESUMO_PATH = os.path.join(RESULTS_DIR, "parte3_resumo.md")
with open(RESUMO_PATH, "r", encoding="utf-8") as f:
    resumo_texto = f.read()

print("=== RASCUNHOS FINAIS DAS 3 PERGUNTAS DA PARTE III ===")
for bloco in resumo_texto.split("### Rascunho da Resposta"):
    bloco_strip = bloco.strip()
    if bloco_strip.startswith("(Pergunta"):
        partes_traco = bloco_strip.split("---")
        corpo_pergunta = partes_traco[0].strip()
        linhas = [l.strip() for l in corpo_pergunta.split("\n") if l.strip()]
        header = linhas[0]
        texto = " ".join(linhas[1:])
        print(f"\n{header}\n{texto}\n")